# Slices y sesgo: el promedio miente

**Facsímil 8 · La ciencia de los datos** — capítulo 5 (slices, sesgos y decisión algorítmica).

Un acierto global estupendo puede esconder que tu modelo funciona **fatal** con un grupo concreto. El
número global —ese 90% que tranquiliza— está dominado por la mayoría, y puede tapar que el sistema falla
justo con quien menos representado está. En este cuaderno coges un modelo con buen acierto global, lo
**partes por grupos** (*slices*) y descubres el sesgo que el promedio ocultaba. Luego vas más allá del
acierto: miras *cómo* falla (falsos positivos y negativos), compruebas que el problema empeora cuanto más
pequeño es el grupo, intentas mitigarlo y mides el precio. Si decides sobre personas —créditos,
selección, diagnóstico—, el promedio no te exime: tienes que mirar **a quién estás fallando**.

### Qué vas a aprender
- Por qué un **acierto global alto** puede convivir con un fallo grave en un subgrupo.
- A medir el rendimiento **por *slices*** (por grupo, segmento o tipo de caso).
- Que el acierto no lo dice todo: hay que mirar **cómo** se falla (falsos positivos frente a negativos).
- Que cuanto más **minoritario** es un grupo, más invisible es para el promedio.
- A pensar en **mitigaciones** (reequilibrar), medir su **precio** y comprobar una **regla de equidad**.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves, sin instalar nada raro.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
np.random.seed(0)

# Dos grupos. El A es mayoria (85%) y tiene una relacion clara senal->etiqueta.
# El B es minoria (15%) y su patron es DISTINTO: el modelo, dominado por A, le va mal.
def genera(n, grupo):
    x = np.random.normal(0, 1, (n, 3))
    if grupo == "A":
        y = (x[:,0] + x[:,1] > 0).astype(int)
    else:
        y = (x[:,2] - x[:,0] > 0).astype(int)   # otra regla
    return x, y, np.array([grupo]*n)

xa, ya, ga = genera(1700, "A")
xb, yb, gb = genera(300, "B")
X = np.vstack([xa, xb]); y = np.concatenate([ya, yb]); g = np.concatenate([ga, gb])
idx = np.random.permutation(len(y)); X, y, g = X[idx], y[idx], g[idx]
corte = 1400
Xtr, ytr = X[:corte], y[:corte]
Xte, yte, gte = X[corte:], y[corte:], g[corte:]
print(f"Entrenamiento: {corte} casos (85% grupo A, 15% grupo B, con reglas distintas).")
print(f"Test: {len(yte)} casos. El grupo B es minoria: pesa poco en cualquier promedio.")


## 1. El número que tranquiliza

Entrenamos un modelo y miramos su acierto global. Sale alto, el típico número que aparece en una
presentación y hace asentir a todo el mundo. Pero un número global es un **promedio**, y los promedios
esconden lo que pasa dentro.


In [ ]:
modelo = LogisticRegression().fit(Xtr, ytr)
pred = modelo.predict(Xte)
acc_global = (pred == yte).mean()
print(f"Acierto GLOBAL: {acc_global:.3f}   <- 'el modelo va bien', diria cualquiera")


## 2. Ahora míralo por grupos

El mismo modelo, las mismas predicciones, pero separando el acierto del grupo A y del grupo B. Aquí es
donde el promedio confiesa lo que escondía. Calcular métricas **por *slice*** es la herramienta más
simple y más reveladora de la ciencia de datos responsable.


In [ ]:
print(f"{'grupo':>6} | {'casos':>6} | {'acierto':>8}")
for grupo in ["A", "B"]:
    m = gte == grupo
    print(f"{grupo:>6} | {m.sum():>6} | {(pred[m]==yte[m]).mean():>8.3f}")
print("\nEl global se parecia al del grupo MAYORITARIO. El grupo B, minoria, queda mucho peor servido.")


**Esto es lo que el promedio tapa.** El acierto global se parece sospechosamente al del grupo
mayoritario, porque son la mayoría de los casos y mandan en la media. El grupo B, pequeño, casi no mueve
el número global... pero recibe un servicio mucho peor. Si ese grupo son personas, acabas de descubrir
un sesgo que ninguna métrica global te iba a enseñar.


## 3. El acierto no lo dice todo: mira *cómo* se falla

Dos modelos con el mismo acierto pueden equivocarse de formas muy distintas, y la diferencia importa
muchísimo. No es lo mismo **denegar un crédito a quien lo pagaría** (falso negativo) que **concederlo a
quien no lo devolverá** (falso positivo): el daño recae sobre personas distintas. Vamos a desglosar, por
grupo, qué tipo de error comete el modelo. El acierto solo cuenta cuántos fallos hay; esto cuenta de qué
clase son.


In [ ]:
print(f"{'grupo':>6} | {'falsos positivos':>16} | {'falsos negativos':>16}")
for grupo in ["A", "B"]:
    m = gte == grupo
    fp = ((pred[m] == 1) & (yte[m] == 0)).mean()   # dice "si" cuando era "no"
    fn = ((pred[m] == 0) & (yte[m] == 1)).mean()    # dice "no" cuando era "si"
    print(f"{grupo:>6} | {fp:>15.1%} | {fn:>15.1%}")
print("\nNo solo falla MAS con el grupo B: ademas falla de forma distinta. Un mismo 'acierto' puede")
print("repartir el dano de maneras muy diferentes entre grupos. Por eso no basta con un numero.")


## 4. Por qué pasa: la minoría no mueve el promedio

Cuanto más pequeño es un grupo, menos pesa su (mal) rendimiento en el número global, y más invisible se
vuelve. Lo medimos: hacemos el grupo B cada vez más minoritario y vemos cómo su mal acierto apenas mueve
el global, aunque para esas personas el sistema siga siendo igual de malo.


In [ ]:
def experimento(frac_B):
    nB = int(2000 * frac_B); nA = 2000 - nB
    xa, ya, _ = genera(nA, "A"); xb, yb, _ = genera(nB, "B")
    X = np.vstack([xa, xb]); y = np.concatenate([ya, yb]); gg = np.array(["A"]*nA + ["B"]*nB)
    idx = np.random.permutation(len(y)); X, y, gg = X[idx], y[idx], gg[idx]
    c = 1600; m = LogisticRegression().fit(X[:c], y[:c])
    pr = m.predict(X[c:]); yt, gt = y[c:], gg[c:]
    glob = (pr==yt).mean()
    accB = (pr[gt=="B"]==yt[gt=="B"]).mean() if (gt=="B").sum() else float("nan")
    return glob, accB

print("% grupo B | acierto GLOBAL | acierto del grupo B")
for frac in [0.40, 0.30, 0.15, 0.05]:
    glob, accB = experimento(frac)
    print(f"   {frac*100:>2.0f}%    |     {glob:.3f}     |       {accB:.3f}")
print("\nCuanto mas pequeno es B, MENOS afecta su mal acierto al global: mas invisible para el promedio.")


## 5. Un tercer grupo todavía más pequeño

Para ver hasta dónde llega la invisibilidad, añadimos un grupo C, muy minoritario (5%) y con **otra
regla más**. El modelo, dominado por A, tampoco lo aprende. La pregunta del experimento: ¿el acierto
global se entera de que C existe?


In [ ]:
def genera3(n, grupo):
    x = np.random.normal(0, 1, (n, 3))
    if grupo == "A":
        y = (x[:,0] + x[:,1] > 0).astype(int)
    elif grupo == "B":
        y = (x[:,2] - x[:,0] > 0).astype(int)
    else:                                   # grupo C: una tercera regla
        y = (x[:,1] - x[:,2] > 0).astype(int)
    return x, y, np.array([grupo]*n)

partes = [genera3(1500, "A"), genera3(400, "B"), genera3(100, "C")]
X3 = np.vstack([p[0] for p in partes])
y3 = np.concatenate([p[1] for p in partes])
g3 = np.concatenate([p[2] for p in partes])
idx = np.random.permutation(len(y3)); X3, y3, g3 = X3[idx], y3[idx], g3[idx]
c = 1600
mod3 = LogisticRegression().fit(X3[:c], y3[:c])
pr3 = mod3.predict(X3[c:]); yt3, gt3 = y3[c:], g3[c:]
print(f"Acierto GLOBAL (3 grupos): {(pr3==yt3).mean():.3f}")
print(f"{'grupo':>6} | {'casos':>6} | {'acierto':>8}")
for grupo in ["A", "B", "C"]:
    m = gt3 == grupo
    print(f"{grupo:>6} | {m.sum():>6} | {(pr3[m]==yt3[m]).mean():>8.3f}")
print("\nEl global sigue alto y tranquilizador, pero ni B ni C asoman: dos grupos peor servidos, invisibles")
print("en el promedio. Cuanto mas pequeno es el grupo, menos pesa su mal rendimiento en el numero global.")


## 6. Mitigar: reequilibrar (y su precio)

Una vez visto el sesgo, ¿se puede corregir? Una opción: dar **más peso** al grupo minoritario al
entrenar, para que el modelo no lo ignore. Casi siempre hay una **balanza**: mejorar el grupo B puede
empeorar un poco el A. Lo importante es que esa decisión sea **explícita**, no un accidente del promedio.


In [ ]:
# damos mas peso a los casos del grupo B durante el entrenamiento
peso = np.where(g[:corte] == "B", 5.0, 1.0)
modelo_eq = LogisticRegression().fit(Xtr, ytr, sample_weight=peso)
pred_eq = modelo_eq.predict(Xte)
print(f"{'grupo':>6} | acierto SIN reequilibrar | acierto CON reequilibrar")
for grupo in ["A", "B"]:
    m = gte == grupo
    a0 = (pred[m]==yte[m]).mean(); a1 = (pred_eq[m]==yte[m]).mean()
    print(f"{grupo:>6} |          {a0:.3f}          |          {a1:.3f}")
print(f"\nGLOBAL  |          {acc_global:.3f}          |          {(pred_eq==yte).mean():.3f}")
print("Reequilibrar suele subir al grupo B a costa de algo del A (y del global). La balanza se decide a proposito.")


## 7. ¿Cuánto peso conviene? Recorriendo la balanza

El peso 5 fue una elección arbitraria. Vamos a recorrer varios valores y ver el compromiso completo:
cómo sube B y cómo baja A a medida que insistimos más en el grupo minoritario. No hay un número
«correcto» universal; hay una frontera que **tú** decides según lo que cueste cada tipo de error.


In [ ]:
print(f"{'peso a B':>9} | {'acierto A':>10} | {'acierto B':>10} | {'brecha A-B':>11}")
for w in [1.0, 2.0, 5.0, 10.0, 20.0]:
    peso = np.where(g[:corte] == "B", w, 1.0)
    mod = LogisticRegression().fit(Xtr, ytr, sample_weight=peso)
    pr = mod.predict(Xte)
    aA = (pr[gte=="A"]==yte[gte=="A"]).mean()
    aB = (pr[gte=="B"]==yte[gte=="B"]).mean()
    print(f"{w:>9.0f} | {aA:>10.3f} | {aB:>10.3f} | {abs(aA-aB):>11.3f}")
print("\nMas peso a B estrecha la brecha... hasta cierto punto, y a costa de A. La frontera se elige, no se hereda.")


## 8. Una regla de equidad explícita

«Que no haya sesgo» suena bien pero no se puede medir. En la práctica se fija un **criterio concreto**,
por ejemplo: *el acierto no debe diferir más de 10 puntos entre grupos*. Eso ya es comprobable: pasa o
no pasa. Lo aplicamos al modelo sin reequilibrar y al reequilibrado, para ver cuál cumple la regla.


In [ ]:
UMBRAL = 0.10
def cumple(pred_, etiqueta):
    aA = (pred_[gte=="A"]==yte[gte=="A"]).mean()
    aB = (pred_[gte=="B"]==yte[gte=="B"]).mean()
    brecha = abs(aA - aB)
    estado = "CUMPLE" if brecha <= UMBRAL else "NO CUMPLE"
    print(f"{etiqueta:>22}: brecha = {brecha:.3f}  ->  {estado} (umbral {UMBRAL:.2f})")

cumple(pred, "sin reequilibrar")
cumple(pred_eq, "reequilibrado (peso 5)")
print("\nLa equidad deja de ser una intencion y pasa a ser una prueba que se aprueba o se suspende.")


## 9. Pruébalo tú

1. **Cambia el umbral de equidad** de la sección 8 (prueba 0,05 o 0,20): ¿qué modelos pasan? La regla la
   pones tú, y eso ya es una decisión de producto.
2. **Ajusta el peso** en la sección 7 hasta que la brecha baje del umbral: ¿cuánto acierto del grupo A
   sacrificas para conseguirlo?
3. **Haz el grupo B aún más raro** (2%) en la sección 4: ¿hasta qué punto el global deja de enterarse?
4. **Mira los falsos positivos y negativos** (sección 3) en el modelo *reequilibrado*: ¿mejora solo el
   acierto, o también cambia el tipo de error que comete con cada grupo?
5. **Invierte las proporciones:** haz que B sea la mayoría. ¿Ahora a quién abandona el promedio?


## 10. Errores comunes

- **Reportar solo el número global.** Esconde a quién estás fallando. Mide siempre por *slices*
  relevantes.
- **Quedarse en el acierto.** Dos modelos con el mismo acierto pueden repartir el daño de formas muy
  distintas. Mira también *cómo* falla (falsos positivos frente a negativos).
- **Olvidar los grupos pequeños.** Cuanto más minoritarios, más invisibles para el promedio, y más fácil
  ignorarlos sin querer.
- **Creer que el sesgo se arregla solo mirándolo.** No: hay que decidir una mitigación, fijar una regla
  de equidad comprobable y aceptar el precio. Pero no puedes arreglar lo que no mides.
- **Confundir acierto con justicia.** Un modelo «bueno de media» puede ser injusto para un colectivo.


## 11. Qué te llevas

- Un **acierto global alto** puede convivir con un fallo grave en un subgrupo: el promedio está dominado
  por la mayoría.
- **Medir por *slices*** (por grupo, segmento o tipo de caso) es la forma de ver a quién estás fallando,
  la pregunta que de verdad importa cuando decides sobre personas.
- El acierto no basta: importa **cómo** se falla, porque cada tipo de error daña a alguien distinto.
- El sesgo no se arregla solo mirándolo, pero **no puedes arreglar lo que no mides**. Reequilibrar tiene
  un precio, conviene fijar una **regla de equidad** comprobable y decidir la balanza a propósito.

**Para seguir:** el facsímil 9 (seguridad, privacidad y gobernanza) convierte esto en obligación: medir
el impacto por grupos es parte de tratar la IA de forma responsable y auditable.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*